<a href="https://colab.research.google.com/github/eduardoddddddd/superastro/blob/master/Copia_de_astro_horaria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌙 Calculadora Astrológica — Swiss Ephemeris
**Ejecuta primero la Celda 1 (instalación) y luego la Celda 2 (carga de funciones).**  
Después ya puedes usar `horaria()` y `natal()` libremente.

> ⚠️ En cada nueva sesión de Colab hay que volver a ejecutar las celdas 1 y 2.

In [ ]:
# ── CELDA 1: INSTALACIÓN ──────────────────────────────────────────────────────
# Ejecutar solo una vez por sesión
!pip install pyswisseph -q
print('✅ pyswisseph instalado')

In [ ]:
# ── CELDA 2: FUNCIONES BASE ───────────────────────────────────────────────────
# Ejecutar una vez por sesión tras la instalación

import swisseph as swe
from datetime import datetime, timezone

swe.set_ephe_path(None)

LAT_DEFAULT = 36.8878
LON_DEFAULT = -3.4018

SIGNS = ['Aries','Tauro','Géminis','Cáncer','Leo','Virgo',
         'Libra','Escorpio','Sagitario','Capricornio','Acuario','Piscis']

SIGN_RULERS = {
    'Aries':'Marte','Tauro':'Venus','Géminis':'Mercurio','Cáncer':'Luna',
    'Leo':'Sol','Virgo':'Mercurio','Libra':'Venus','Escorpio':'Marte',
    'Sagitario':'Júpiter','Capricornio':'Saturno','Acuario':'Saturno','Piscis':'Júpiter'
}

EXALTATIONS = {
    'Sol':'Aries','Luna':'Tauro','Mercurio':'Virgo','Venus':'Piscis',
    'Marte':'Capricornio','Júpiter':'Cáncer','Saturno':'Libra'
}

DETRIMENT = {
    'Sol':'Acuario','Luna':'Capricornio','Mercurio':['Sagitario','Piscis'],
    'Venus':['Aries','Escorpio'],'Marte':['Libra','Tauro'],
    'Júpiter':'Géminis','Saturno':'Cáncer'
}

FALL = {
    'Sol':'Libra','Luna':'Escorpio','Mercurio':'Piscis','Venus':'Virgo',
    'Marte':'Cáncer','Júpiter':'Capricornio','Saturno':'Aries'
}

PLANET_IDS = {
    'Sol': swe.SUN, 'Luna': swe.MOON, 'Mercurio': swe.MERCURY,
    'Venus': swe.VENUS, 'Marte': swe.MARS, 'Júpiter': swe.JUPITER,
    'Saturno': swe.SATURN
}

def deg_to_sign_pos(deg):
    deg = deg % 360
    s = int(deg / 30)
    d = deg % 30
    m = (d % 1) * 60
    return SIGNS[s], int(d), int(m), s

def fmt(deg):
    sign, d, m, _ = deg_to_sign_pos(deg)
    return f"{d:02d}°{sign[:3]}{m:02d}'"

def which_house(planet_lon, cusps):
    for i in range(12):
        start = cusps[i] % 360
        end   = cusps[(i+1) % 12] % 360
        lon   = planet_lon % 360
        if start <= end:
            if start <= lon < end:
                return i + 1
        else:
            if lon >= start or lon < end:
                return i + 1
    return 1

def dignity(planet_name, sign_name):
    rulers_by_sign = {s: p for s, p in SIGN_RULERS.items()}
    if rulers_by_sign.get(sign_name) == planet_name:
        return 'DOMICILIO'
    if EXALTATIONS.get(planet_name) == sign_name:
        return 'EXALTACIÓN'
    det = DETRIMENT.get(planet_name, [])
    if isinstance(det, str): det = [det]
    if sign_name in det:
        return 'DETRIMENTO'
    if FALL.get(planet_name) == sign_name:
        return 'CAÍDA'
    return ''

def aspect_name(angle):
    aspects = {0:'Conjunción', 60:'Sextil', 90:'Cuadratura',
               120:'Trígono', 180:'Oposición'}
    for deg, name in aspects.items():
        if abs(angle - deg) <= 8 or abs(angle - (360-deg)) <= 8:
            return name, round(min(abs(angle-deg), abs(angle-(360-deg))), 2)
    return None, None


def horaria(lat=LAT_DEFAULT, lon=LON_DEFAULT, fecha=None):
    """
    Carta horaria.
    horaria()                            → ahora, Órgiva
    horaria(lat=40.416, lon=-3.703)      → ahora, Madrid
    horaria(fecha=(2026,3,26,10,0))      → fecha manual (año,mes,día,hora_local,min)
    """
    if fecha is None:
        now = datetime.now(timezone.utc)
        Y, M, D = now.year, now.month, now.day
        H_utc = now.hour + now.minute/60 + now.second/3600
        hora_str = now.strftime('%Y-%m-%d %H:%M UTC')
    else:
        Y, M, D, h_local, m_local = fecha
        H_utc = (h_local - 1) + m_local/60
        hora_str = f"{Y}-{M:02d}-{D:02d} {h_local:02d}:{m_local:02d} CET"

    JD = swe.julday(Y, M, D, H_utc)
    cusps, ascmc = swe.houses(JD, lat, lon, b'R')
    ASC = ascmc[0]
    MC  = ascmc[1]

    print(f"\n{'='*60}")
    print(f"  CARTA HORARIA")
    print(f"  {hora_str}")
    print(f"  {lat}°N {abs(lon)}°{'O' if lon<0 else 'E'}  |  Regiomontanus")
    print(f"{'='*60}")
    print(f"\n  ASC: {fmt(ASC)}    MC: {fmt(MC)}")
    asc_sign = SIGNS[int(ASC/30)]
    print(f"  Señor Consultante (Casa 1): {SIGN_RULERS[asc_sign]}")

    print(f"\n{'─'*60}")
    print(f"  CÚSPIDES")
    print(f"{'─'*60}")
    for i, c in enumerate(cusps):
        s = SIGNS[int(c/30)]
        print(f"  Casa {i+1:2d}: {fmt(c)}   → Señor: {SIGN_RULERS[s]}")

    positions = {}
    speeds    = {}
    print(f"\n{'─'*60}")
    print(f"  PLANETAS")
    print(f"{'─'*60}")
    for name, pid in PLANET_IDS.items():
        res  = swe.calc_ut(JD, pid)
        plon = res[0][0]
        spd  = res[0][3]
        positions[name] = plon
        speeds[name]    = spd
        rx      = ' Rx' if spd < 0 else '   '
        sign, d, m, _ = deg_to_sign_pos(plon)
        casa    = which_house(plon, cusps)
        dig     = dignity(name, sign)
        dig_str = f'  [{dig}]' if dig else ''
        print(f"  {name:<10} {fmt(plon)}{rx}  Casa {casa:2d}  vel:{spd:+.4f}{dig_str}")

    nn = swe.calc_ut(JD, swe.TRUE_NODE)
    positions['NodoN'] = nn[0][0]
    casa_nn = which_house(nn[0][0], cusps)
    print(f"  {'Nodo N':<10} {fmt(nn[0][0])}     Casa {casa_nn:2d}")

    print(f"\n{'─'*60}")
    print(f"  ASPECTOS DE LUNA")
    print(f"{'─'*60}")
    luna = positions['Luna']
    for name, plon in positions.items():
        if name == 'Luna': continue
        angle = abs(luna - plon) % 360
        if angle > 180: angle = 360 - angle
        asp, orbe = aspect_name(angle)
        if asp:
            aplicativo = 'aplicativo' if (luna < plon and speeds.get('Luna',0) > 0) else 'separativo'
            print(f"  Luna {asp} {name:<10} orbe {orbe:5.2f}°  ({aplicativo})")

    print(f"\n{'='*60}\n")
    return positions, cusps, speeds


def natal(year=1976, month=10, day=11, hour_local=20, minute=33,
          lat=40.416, lon=-3.703, sistema=b'P'):
    """
    Carta natal. Por defecto: Abdul Malik, Madrid, Placidus.
    sistema: b'P' = Placidus, b'R' = Regiomontanus
    """
    H_utc = (hour_local - 1) + minute/60
    JD    = swe.julday(year, month, day, H_utc)
    cusps, ascmc = swe.houses(JD, lat, lon, sistema)
    ASC = ascmc[0]
    MC  = ascmc[1]
    sis_name = 'Placidus' if sistema == b'P' else 'Regiomontanus'

    print(f"\n{'='*60}")
    print(f"  CARTA NATAL — {day}/{month}/{year} {hour_local:02d}:{minute:02d} CET")
    print(f"  {lat}°N {abs(lon)}°{'O' if lon<0 else 'E'}  |  {sis_name}")
    print(f"{'='*60}")
    print(f"\n  ASC: {fmt(ASC)}    MC: {fmt(MC)}")

    print(f"\n{'─'*60}")
    print(f"  PLANETAS")
    print(f"{'─'*60}")
    for name, pid in PLANET_IDS.items():
        res  = swe.calc_ut(JD, pid)
        plon = res[0][0]
        spd  = res[0][3]
        rx   = ' Rx' if spd < 0 else '   '
        sign, d, m, _ = deg_to_sign_pos(plon)
        casa = which_house(plon, cusps)
        dig  = dignity(name, sign)
        dig_str = f'  [{dig}]' if dig else ''
        print(f"  {name:<10} {fmt(plon)}{rx}  Casa {casa:2d}{dig_str}")

    nn = swe.calc_ut(JD, swe.TRUE_NODE)
    casa_nn = which_house(nn[0][0], cusps)
    print(f"  {'Nodo N':<10} {fmt(nn[0][0])}     Casa {casa_nn:2d}")
    print(f"\n{'='*60}\n")
    return cusps, ascmc


print('✅ Funciones cargadas: horaria() | natal()')
print('   horaria()                         → ahora, Órgiva')
print('   horaria(fecha=(2026,3,26,10,0))   → fecha manual')
print('   natal()                           → carta natal Abdul Malik')

In [ ]:
# ── CELDA 3: CARTA HORARIA AHORA ─────────────────────────────────────────────
horaria()

In [ ]:
# ── CELDA 4: CARTA HORARIA CON FECHA MANUAL ───────────────────────────────────
# Modifica los valores según necesites: (año, mes, día, hora_local, minutos)
horaria(fecha=(2026, 3, 26, 10, 0))

In [ ]:
# ── CELDA 5: CARTA NATAL (por defecto Abdul Malik) ────────────────────────────
natal()

In [ ]:
# ── CELDA LIBRE: escribe aquí tus consultas ad hoc ───────────────────────────
